# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}\n")
print("Keywords:", ', '.join(metadata.keywords))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available RecordSets and Fields by their '@id'

record_sets = dataset.metadata.recordSet

if not record_sets or len(record_sets) == 0:
    print("No record sets found directly in the root of metadata. Searching in distribution...")
    # Sometimes record sets are located in the distribution
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            if hasattr(dist, 'recordSet'):
                record_sets = dist.recordSet
                break

if not record_sets or len(record_sets) == 0:
    print('No record sets identified in either root or distribution.')
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        fields = rs['field']
        print("Fields:")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"  - Field @id: {field_id}")
        print('-' * 40)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we need to examine the structure for record set IDs.
# If there are no record sets in the root, try to extract from metadata.distribution

# Search for record set IDs
record_set_ids = []

def extract_recordsets_from_metadata(md):
    found = False
    if hasattr(md, 'recordSet') and md.recordSet:
        for rs in md.recordSet:
            if isinstance(rs, dict) and '@id' in rs:
                record_set_ids.append(rs['@id'])
                found = True
    if not found and hasattr(md, 'distribution'):
        # Some datasets store recordSets inside distributions
        for dist in md.distribution:
            if hasattr(dist, 'recordSet') and dist.recordSet:
                for rs in dist.recordSet:
                    if isinstance(rs, dict) and '@id' in rs:
                        record_set_ids.append(rs['@id'])

extract_recordsets_from_metadata(metadata)

if len(record_set_ids) == 0:
    print("No record sets found. Please check the Croissant schema.")
else:
    print(f"Record sets found: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields in {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Suppose our main data is in the first record set
import numpy as np

if len(dataframes) > 0:
    main_record_set_id = record_set_ids[0]
    df_main = dataframes[main_record_set_id]
    # Try to auto-detect a likely numeric field (for mass spectrometry data: 'MW', 'Abundance', 'Coverage', etc.)
    # Use columns with float or int dtype
    numeric_cols = df_main.select_dtypes(include=np.number).columns.tolist()
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
        threshold = df_main[numeric_field].mean() if df_main[numeric_field].mean() > 0 else 10
        filtered_df = df_main[df_main[numeric_field] > threshold]
        print(f"Filtered records in [{main_record_set_id}] where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Group by first non-numeric field, if any
        group_cols = [c for c in df_main.columns if c not in numeric_cols]
        if group_cols:
            group_field = group_cols[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
        else:
            print('No non-numeric field found for grouping.')
    else:
        print('No numeric fields available for analysis.')
else:
    print('No dataframes loaded in previous steps. Cannot perform EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If we found numeric_field and dataframe
if len(dataframes) > 0 and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of `{numeric_field}`')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # If grouped_df from above
    if 'grouped_df' in locals():
        grouped_df.plot(kind='bar', figsize=(10,5), title=f'Mean {numeric_field} by {group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.show()
else:
    print('No numeric data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated the step-by-step process of loading, exploring, and analyzing the FAIR^2 dataset using `mlcroissant` and `pandas`.
- We inspected record sets, loaded records by their `@id`, and performed basic EDA, such as filtering on numeric fields and visualizing distributions.
- For deeper scientific analysis, see the fields explored and consult the original dataset's documentation and Croissant schema.